# End-to-End Forensic Intelligence Pipeline Demo
## From Raw File Fragments to Validated File Reconstruction

This interactive notebook demonstrates our complete proposed forensic recovery workflow:
$$\text{Raw Fragment} \xrightarrow{\text{Byte2Image}} \text{2D Grayscale} \xrightarrow{\text{Swin V2 Tiny}} \text{Type Probabilities} \xrightarrow{\text{Graph Reassembly}} \text{Reconstructed File} \xrightarrow{\text{Validator}} \text{Forensic Proof}$$

In [ ]:
import os, sys, io, zipfile, json
sys.path.insert(0, os.path.abspath(".."))

import torch
from src.models.classifier import FragmentClassifier
from src.representations.byte2image import bytes_to_byte2image
from src.reassembly.graph import FragmentNode
from src.reassembly.edge_scoring import build_complete_reassembly_graph
from src.reassembly.ordering import order_fragments_beam_search
from src.reassembly.reconstruct import reconstruct_file_from_ordering
from src.validation.validator import validate_reconstructed_file
import matplotlib.pyplot as plt

print("Complete Forensic Pipeline Modules Loaded!")

In [ ]:
# 1. Create a Test Multi-Fragment File (ZIP Archive)
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("confidential.txt", "Recovered forensic evidence from deleted XFS filesystem block clusters.")
    zf.writestr("meta.json", json.dumps({"timestamp": "2026-08-26", "case": "CR-4029"}))
original_data = buf.getvalue()

# 2. Split into 512-byte fragments & shuffle
chunk_size = 512
chunks = [original_data[i:i+chunk_size].ljust(chunk_size, b'\x00') for i in range(0, len(original_data), chunk_size)]
print(f"Original file size: {len(original_data)} bytes -> {len(chunks)} fragments")

# 3. Run Byte2Image + Classifier on each fragment
classifier = FragmentClassifier(device="cuda" if torch.cuda.is_available() else "cpu")
fragment_nodes = []

for idx, chunk in enumerate(chunks):
    pred = classifier.predict_fragment(chunk)
    node = FragmentNode(
        fragment_id=f"frag_{idx:02d}",
        raw_bytes=chunk,
        predicted_class=pred["predicted_class"],
        confidence=pred["confidence"],
        probabilities=pred["all_probabilities"]
    )
    fragment_nodes.append(node)
    print(f"Fragment {idx}: Predicted={pred['predicted_class'].upper()} (Confidence={pred['confidence']:.2%})")

In [ ]:
# 4. Graph Construction, Ordering, and Reconstruction
graph = build_complete_reassembly_graph(fragment_nodes)
ordered_ids, score, conf, unused = order_fragments_beam_search(graph, beam_width=5)
recon = reconstruct_file_from_ordering(ordered_ids, graph, score, conf, unused)

print("=" * 60)
print(f"Reconstructed Sequence:  {' -> '.join(ordered_ids)}")
print(f"Reconstruction Size:     {recon.total_size} bytes")
print(f"SHA-256 Hash:            {recon.sha256}")
print("=" * 60)

In [ ]:
# 5. Deep Format Validation
report = validate_reconstructed_file(recon.reconstructed_bytes, expected_format="zip")
print(f"Validation Status:  {report.status}")
print(f"Structural Valid:   {report.is_valid}")
print(f"Details:            {json.dumps(report.details, indent=2)}")